# 🚀 Google Gemini SDK: Beginner to Intermediate Guide

Welcome to the hands-on Google Gemini tutorial! This notebook will take you from sending your first API request to constructing an **Agent with Function Calling (Tool Use)**.

### Setup Prerequisites
Before starting, make sure you have installed the modern `google-genai` SDK and set your API key.

```bash
pip install google-genai pydantic python-dotenv

In [1]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load API key from environment or .env file
load_dotenv()

# Initialize the client (automatically uses GEMINI_API_KEY from environment)
client = genai.Client()

## Level 1: Basic Text Generation (Hello World)

**Concept:** The simplest way to interact with Gemini is by sending a single prompt to `client.models.generate_content()`.

We'll use `gemini-3.5-flash` as our default model for fast, standard tasks.

In [3]:
# Level 1 Example: Basic Text Generation
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explain what an LLM is in 2 concise sentences."
)

print("--- Level 1 Response ---")
print(response.text)

--- Level 1 Response ---
A Large Language Model (LLM) is an artificial intelligence system trained on massive amounts of text data to understand, process, and generate human-like language. By predicting the most likely next words in a sequence, it can perform diverse tasks such as answering questions, translating languages, and writing creative content.


## Level 2: System Instructions & Configurations

**Concept:** You can influence Gemini's personality, creativity, and tone using `GenerateContentConfig`.
- `system_instruction`: Dictates the persona or strict rules the model must follow.
- `temperature`: Controls randomness (0.0 = deterministic/factual, 1.0 = creative).

In [4]:
# Level 2 Example: Custom System Prompt & Low Temperature
config = types.GenerateContentConfig(
    system_instruction="You are a strict, highly accurate math and logic tutor. Be concise.",
    temperature=0.1
)

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Is 107 a prime number? Show quick logic.",
    config=config
)

print("--- Level 2 Response ---")
print(response.text)

--- Level 2 Response ---
**Yes, 107 is a prime number.**

**Logic:**
1. Find the approximate square root: $\sqrt{107} \approx 10.3$.
2. Test divisibility by prime numbers less than 10.3 (2, 3, 5, and 7):
   * **2:** No (ends in an odd digit).
   * **3:** No (sum of digits $1+0+7 = 8$, which is not divisible by 3).
   * **5:** No (does not end in 0 or 5).
   * **7:** No ($107 \div 7 = 15$ with a remainder of 2).

Since 107 is not divisible by any prime up to its square root, it is prime.


## Level 3: Enforcing Structured JSON Output

**Concept:** Instead of unstructured text, you can enforce Gemini to output pure, type-safe JSON schema using Python's `pydantic`.

In [5]:
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# -------------------------------------------------------------------------
# Step 1: Define your target output structure using Pydantic
# -------------------------------------------------------------------------
# Inheriting from BaseModel tells Pydantic that this class represents a structured data model.
class Flashcard(BaseModel):
    # Field(description=...) gives Gemini semantic context so it knows what to populate in each field.
    term: str = Field(description="The word or concept name")
    definition: str = Field(description="A 1-sentence easy definition")
    example: str = Field(description="A real-world example")


# -------------------------------------------------------------------------
# Step 2: Configure Gemini's output generation behavior
# -------------------------------------------------------------------------
json_config = types.GenerateContentConfig(
    # Sets the output format to JSON instead of default free-form plain text/markdown.
    response_mime_type="application/json",
    
    # Enforces strict adherence to our Pydantic schema structure.
    # Gemini will dynamically convert this Python class into a JSON Schema.
    response_schema=Flashcard,
    
    # Controls answer randomness: 0.0 is deterministic and factual; higher values (e.g., 0.8) are more creative.
    temperature=0.2,
    
    # Explicitly disables the Automatic Function Calling (AFC) handler in the SDK,
    # preventing internal warnings when generating structured schemas without calling functions.
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
)


# -------------------------------------------------------------------------
# Step 3: Execute the request
# -------------------------------------------------------------------------
response = client.models.generate_content(
    # Specifies which model variant to run (e.g., gemini-2.5-flash or gemini-3.5-flash-lite).
    model="gemini-3.5-flash",
    
    # The prompt containing the text instructions for Gemini to analyze.
    contents="Generate a study flashcard for the concept of 'Recursion'.",
    
    # Applies our defined JSON schema and temperature configurations to this execution.
    config=json_config
)

# Print the structured JSON string returned by Gemini
print("--- Level 3 Output (Structured JSON) ---")
print(response.text)

--- Level 3 Output (Structured JSON) ---
{"term": "Recursion", "definition": "Recursion is a programming technique where a function calls itself to solve a smaller instance of the same problem until it reaches a base case.", "example": "A classic example of recursion is calculating the factorial of a number, where factorial(n) is defined as n multiplied by factorial(n-1), stopping when n is 1."}


## Level 4: Multi-Turn Chat (Conversational Memory)

**Concept:** Instead of manually tracking conversation history, use `client.chats.create()`. The chat object automatically remembers prior messages.

In [6]:
# Level 4 Example: Starting a chat session
chat = client.chats.create(model="gemini-3.5-flash")

# Message 1
response_1 = chat.send_message("My favorite programming language is Python.")
print("Bot:", response_1.text)

# Message 2 (Gemini remembers the context)
response_2 = chat.send_message("What is my favorite language?")
print("\nBot:", response_2.text)

Bot: Python is an excellent choice! It is one of the most popular and versatile programming languages in the world. 

There is so much to love about it, including:
* **Readability:** Its clean syntax almost feels like reading English, making it a joy to write and debug.
* **The Ecosystem:** With libraries like Pandas, NumPy, TensorFlow, Django, and Flask, you can build almost anything.
* **Versatility:** It's the go-to language for AI, machine learning, data science, web development, automation, and scripting.

What do you enjoy building the most with Python? Are you into data science, web development, automation, or something else entirely?

Bot: Your favorite language is **Python**! You just told me a moment ago. 😉


## Level 5 (Intermediate): Tool Use / Function Calling

**Concept:** LLMs cannot execute code or access live data directly. However, Gemini can detect when it needs external information, pause execution, and request your code to execute a Python function on its behalf!

Using Automatic Function Calling (AFC) in `client.chats`, Gemini will automatically call local Python functions and weave their return values back into its final response.

In [7]:
# 1. Define a standard Python function with type hints and docstrings
def get_stock_price(symbol: str) -> str:
    """Returns the current stock price for a given ticker symbol.
    
    Args:
        symbol: The stock ticker symbol (e.g., GOOGL, AAPL).
    """
    # Mocking database / external API call
    mock_db = {
        "GOOGL": "$185.50",
        "AAPL": "$225.10",
        "MSFT": "$450.00"
    }
    symbol_upper = symbol.upper()
    price = mock_db.get(symbol_upper, "Price unavailable")
    return f"The current price for {symbol_upper} is {price}."

# 2. Pass the function directly into the chat configuration's `tools` array
agent_chat = client.chats.create(
    model="gemini-3.5-flash",
    config=types.GenerateContentConfig(
        tools=[get_stock_price],  # Registering our python function as a tool
        temperature=0.0
    )
)

# 3. Query the agent with a question that requires external data lookup
print("--- Level 5 Agent Processing ---")
user_query = "Hey, what is the current price of AAPL stock?"
response = agent_chat.send_message(user_query)

print("Final Agent Response:")
print(response.text)

--- Level 5 Agent Processing ---
Final Agent Response:
The current price of AAPL stock is $225.10.


# Lesson 5 (Part 2 with Tavily)
```bash
pip install google-genai tavily-python

In [22]:
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [23]:
# -------------------------------------------------------------------------
# Step 1: Define the Python Tool Function
# -------------------------------------------------------------------------
def search_web(query: str) -> dict:
    """Performs a live web search using Tavily to retrieve current news and information.
    
    Args:
        query: The search query string (e.g., 'latest AI breakthroughs this week').
    """
    print(f"\n[AFC TRIGGERED] Executing Tavily search for query: '{query}'...")
    
    # Run Tavily search returning clean text snippets
    response = tavily_client.search(query=query, search_depth="basic", max_results=3)
    
    # Structure the extracted fields for Gemini
    results = [
        {
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content")
        }
        for r in response.get("results", [])
    ]
    return {"results": results}

# -------------------------------------------------------------------------
# Step 2: Initialize Chat with AFC Enabled
# -------------------------------------------------------------------------
# Passing `tools=[search_web]` to a chat object enables Automatic Function Calling (AFC).
# The SDK handles the multi-turn loop automatically.
agent_chat = client.chats.create(
    model="gemini-3.6-flash",
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are a real-time news and research assistant. Use the search_web tool "
            "to look up current events and live data. Always include source URLs in your answer."
        ),
        # Pass the python function directly as a tool
        tools=[search_web],
        temperature=0.0
    )
)


# -------------------------------------------------------------------------
# Step 3: Run the Query
# -------------------------------------------------------------------------
user_prompt = "What are the latest key updates on AI agent frameworks?"
print(f"User Query: {user_prompt}")

# Send message: Gemini detects search intent, calls search_web(), and returns the final text response.
response = agent_chat.send_message(user_prompt)

print("\n--- Final Agent Response (with Source Citations) ---")
print(response.text)

User Query: What are the latest key updates on AI agent frameworks?

[AFC TRIGGERED] Executing Tavily search for query: 'latest AI agent frameworks updates 2025'...

[AFC TRIGGERED] Executing Tavily search for query: 'Hugging Face smolagents release announcement'...

[AFC TRIGGERED] Executing Tavily search for query: 'Microsoft AutoGen v0.4 release AG2 AI agent frameworks'...


KeyboardInterrupt: 

## Level 6 (Intermediate): Manual Function Calling with Tavily Web Search

**Concept:** Automatic Function Calling (AFC) is great for rapid prototyping, but in production systems you often want **Manual Function Calling**. 

Manual tool calling gives you full control over the execution loop:
1. Gemini decides *if* and *which* tool to invoke, returning a `function_call` payload.
2. Your Python app intercepts the request, runs the local function (e.g., querying Tavily), and validates the results.
3. Your app manually sends the `function_response` back to Gemini to generate the final human-readable answer.

---

### Step 1: Install Tavily and Setup Keys

```bash
pip install google-genai tavily-python python-dotenv

In [13]:
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [14]:
# -------------------------------------------------------------------------
# Step 1: Define the Tavily Search Tool Function
# -------------------------------------------------------------------------
def search_web(query: str) -> dict:
    """Performs a web search using Tavily to get real-time search results.
    
    Args:
        query: The search query string.
    """
    print(f"\n[LOCAL EXECUTION] Running Tavily search for: '{query}'...")
    response = tavily_client.search(query=query, search_depth="basic", max_results=3)
    
    # Extract titles, URLs, and text content for Gemini
    results = [
        {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
        for r in response.get("results", [])
    ]
    return {"results": results}

# Map function names to their Python callable reference
available_tools = {
    "search_web": search_web
}

In [28]:
# -------------------------------------------------------------------------
# Step 2: Configure Gemini with Tool Declarations & Disable AFC
# -------------------------------------------------------------------------
config = types.GenerateContentConfig(
    system_instruction=(
        "You are a real-time web researcher. Use search_web ONCE to gather live facts. "
        "As soon as you receive the search results, immediately synthesize and output your final text answer."
    ),
    tools=[search_web],
    temperature=0.0,
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
)


# -------------------------------------------------------------------------
# Step 3: Turn 1 — Send User Prompt to Gemini
# -------------------------------------------------------------------------
user_prompt = "What are the latest developments in AI agents this week?"
print(f"User Query: {user_prompt}")

# Store the conversation history explicitly
contents = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text=user_prompt)]
    )
]

# Send turn 1 to Gemini
response_turn1 = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=contents,
    config=config
)

User Query: What are the latest developments in AI agents this week?


In [30]:
# -------------------------------------------------------------------------
# Step 4: Detect and Manually Execute Tool Call
# -------------------------------------------------------------------------
# Append model's response to the conversation history
contents.append(response_turn1.candidates[0].content)

# Check if Gemini requested a function call
function_calls = response_turn1.function_calls

if function_calls:
    for call in function_calls:
        function_name = call.name
        function_args = call.args
        
        print(f"\n[GEMINI PROPOSAL] Gemini requested tool: '{function_name}'")
        print(f"[GEMINI ARGS] {function_args}")
        
        # 1. Look up and execute the local python function manually
        if function_name in available_tools:
            tool_result = available_tools[function_name](**function_args)
            
            # 2. Append the function output back into conversation history as a 'function_response' part
            contents.append(
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_function_response(
                            name=function_name,
                            response=tool_result
                        )
                    ]
                )
            )

# -------------------------------------------------------------------------
# Step 5: Turn 2 — Pass Tool Results back to Gemini for Final Answer
# -------------------------------------------------------------------------
response_turn2 = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=contents,
    config=config
)

print("\n--- Final Answer from Gemini ---")
print(response_turn2.text)


[GEMINI PROPOSAL] Gemini requested tool: 'search_web'
[GEMINI ARGS] {'query': 'latest developments in AI agents this week February 2025'}

[LOCAL EXECUTION] Running Tavily search for: 'latest developments in AI agents this week February 2025'...

--- Final Answer from Gemini ---
The landscape of AI agents is shifting rapidly from experimental chatbots to deeply integrated, autonomous "agentic" systems. Key developments and trends center around several core areas:

### 1. The Push Toward Local and Desktop Integration
There is a distinct design pattern shift moving AI agents out of browser windows and directly onto user operating systems. Tech companies and specialized startups are releasing desktop applications designed to act locally on a user's machine—handling cross-application workflows, managing local files, and executing tasks natively (exemplified by recent local desktop agent rollouts and localized OS integration tools). 

### 2. Explosion of Specialized AI Coding Agents
Coding

Markdown
## Level 7 (Intermediate): Multi-Step / Sequential Tool Calling

**Concept:** A single user query often requires multi-step reasoning where the output of Tool 1 serves as the required input for Tool 2. 

For example, asking *"What is the weather in the capital of Nepal?"* requires Gemini to:
1. Call `search_web(query="capital of Nepal")` to discover the capital city (Kathmandu).
2. Receive the result, then call `get_current_weather(city="Kathmandu")`.
3. Synthesize both tool outputs into a final human-readable answer.

---

### Step 1: Install Dependencies

```bash
pip install google-genai tavily-python python-dotenv

In [31]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client()

# -------------------------------------------------------------------------
# Step 1: Define simple sequential tools
# -------------------------------------------------------------------------

def to_lower_case(text: str) -> dict:
    """Converts input text to lowercase.
    
    Args:
        text: The string to transform.
    """
    result = text.lower()
    print(f"\n[TOOL CALL 1] Lowercasing '{text}' -> '{result}'")
    return {"result": result}


def count_characters(text: str) -> dict:
    """Counts the total number of characters in a string.
    
    Args:
        text: The string to analyze.
    """
    length = len(text)
    print(f"\n[TOOL CALL 2] Counting length of '{text}' -> {length} characters")
    return {"character_count": length}


# -------------------------------------------------------------------------
# Step 2: Initialize Chat Agent with AFC (Automatic Function Calling)
# -------------------------------------------------------------------------

agent_chat = client.chats.create(
    model="gemini-3.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are a text execution engine. Execute the tools sequentially in order: "
            "First lower-case the input, then count the resulting characters."
        ),
        tools=[to_lower_case, count_characters],
        temperature=0.0
    )
)

# -------------------------------------------------------------------------
# Step 3: Run the Sequential Prompt
# -------------------------------------------------------------------------

if __name__ == "__main__":
    prompt = "Please convert the string 'HELLO WORLD AI' to lowercase, and then count how many characters it has."
    print(f"User Request: {prompt}")

    # Gemini handles Step 1 -> Step 2 -> Final Answer automatically behind the scenes
    response = agent_chat.send_message(prompt)

    print("\n--- Final Agent Response ---")
    print(response.text)

User Request: Please convert the string 'HELLO WORLD AI' to lowercase, and then count how many characters it has.

[TOOL CALL 1] Lowercasing 'HELLO WORLD AI' -> 'hello world ai'

[TOOL CALL 2] Counting length of 'hello world ai' -> 14 characters

--- Final Agent Response ---
The string 'HELLO WORLD AI' converted to lowercase is 'hello world ai', which contains 14 characters.


# Assignment 1: Weather Agent

Build an AI agent using the **Gemini SDK** that retrieves the current weather for **three specified locations** using OpenWeather. The agent must make the weather API calls **sequentially**, collect the temperature from each location, and calculate and display the **average temperature** across all three locations.

## Grading Rubric — 5 Marks

| Criteria | Marks |
|---|---:|
| Gemini SDK agent implementation | 1 |
| Correct OpenWeather API integration | 1 |
| Sequential tool calls for 3 locations | 1 |
| Correct average temperature calculation | 1 |
| Clear output, error handling & code quality | 1 |
| **Total** | **5** |